# Ερώτημα 1 — Διερευνητική Ανάλυση Δεδομένων

## 1.1 Φόρτωση Δεδομένων
Φόρτωση των 8 CSV αρχείων του CIC-IDS-2017 dataset και ένωσή τους σε ένα ενιαίο DataFrame.

In [ ]:
#Φόρτωση από Google Drive
from google.colab import drive
import zipfile, os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

drive.mount('/content/drive')

#Αποσυμπίεση
zipfile.ZipFile('/content/drive/MyDrive/data_ids.zip').extractall('data/')


In [ ]:
#Φόρτωση & ένωση όλων των CSV
csv_files = glob.glob('data/data_ids/*.csv')

df_list = []
for f in csv_files:
    temp = pd.read_csv(f, low_memory=False)
    temp.columns = temp.columns.str.strip()
    df_list.append(temp)
    print(f" {os.path.basename(f)}: {temp.shape[0]:,} γραμμές")

df = pd.concat(df_list, ignore_index=True)
print(f"\n Συνολικό dataset: {df.shape[0]:,} γραμμές x {df.shape[1]} στήλες")

## 1.2 Δομή & Πληροφορίες Dataset
Εξέταση της δομής του dataset: αριθμός γραμμών/στηλών, τύποι δεδομένων, unique values ανά στήλη και εμφάνιση δειγματικών εγγραφών.

In [ ]:
#Εμφάνιση δομής dataset
print("Πρώτες 5 γραμμές:")
display(df.head())

print("\nΤελευταίες 5 γραμμές:")
display(df.tail())

print("\nΠληροφορίες dataset:")
df.info()

In [ ]:
# Δομή dataset
print("=" * 60)
print("ΔΟΜΗ ΤΟΥ DATASET")
print("=" * 60)
print(f"Γραμμές: {df.shape[0]:,}")
print(f"Στήλες: {df.shape[1]}")

print(f"\nΤύποι δεδομένων:")
print(df.dtypes.value_counts())

print(f"\nΌλες οι στήλες και ο τύπος τους:")
print(df.dtypes)

In [ ]:
# Unique values ανά στήλη
print("Unique values ανά στήλη:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:>2}. {col:<35s}: {df[col].nunique():,}")

## 1.3 Καθαρισμός Δεδομένων
Έλεγχος και διαχείριση ελλιπών τιμών (NaN), άπειρων τιμών (inf) και διπλότυπων εγγραφών. Οι ελλιπείς/άπειρες τιμές αφαιρούνται καθώς αποτελούν αμελητέο ποσοστό (<0.1%) του dataset.

In [ ]:
#Έλεγχος ελλιπών & άπειρων τιμών

# Ελλιπείς τιμές
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Ελλιπείς': missing, 'Ποσοστό (%)': missing_pct.round(4)})
missing_df = missing_df[missing_df['Ελλιπείς'] > 0].sort_values('Ελλιπείς', ascending=False)

if len(missing_df) > 0:
    print("Στήλες με ελλιπείς τιμές:")
    print(missing_df)
else:
    print("Δεν βρέθηκαν ελλιπείς τιμές (NaN).")

#Άπειρες τιμές (inf)
numeric_cols = df.select_dtypes(include=[np.number]).columns
inf_counts = np.isinf(df[numeric_cols]).sum()
inf_cols = inf_counts[inf_counts > 0]

print(f"\nΣτήλες με άπειρες (inf) τιμές:")
if len(inf_cols) > 0:
    print(inf_cols)
else:
    print("Δεν βρέθηκαν άπειρες τιμές.")

In [ ]:
#Οπτικοποίηση ελλιπών τιμών
!pip install missingno -q
import missingno as msno

missing_cols = [col for col in df.columns if df[col].isna().any()]

if len(missing_cols) > 0:
    fig, ax = plt.subplots(figsize=(4, 6))
    msno.bar(df[missing_cols], ax=ax, fontsize=12, color=sns.color_palette('pastel'))
    ax.set_title('Οπτικοποίηση Ελλιπών Τιμών', fontsize=14)
    plt.show()
else:
    print("Δεν υπάρχουν ελλιπείς τιμές για οπτικοποίηση.")
    print("οι ελλιπείς τιμές έχουν ήδη αφαιρεθεί)")

In [ ]:
#Διαχείριση ελλιπών & άπειρων τιμών
print("Πριν τον καθαρισμό:", df.shape[0], "γραμμές")

# Αντικατάσταση inf με NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Αφαίρεση γραμμών με NaN
df.dropna(inplace=True)

print("Μετά τον καθαρισμό:", df.shape[0], "γραμμές")
print(f"Αφαιρέθηκαν: {2830743 - df.shape[0]} γραμμές")

# Επιβεβαίωση
print(f"\nΕλλιπείς τιμές τώρα: {df.isnull().sum().sum()}")
print(f"Άπειρες τιμές τώρα: {np.isinf(df.select_dtypes(include=[np.number])).sum().sum()}")

In [ ]:
#Έλεγχος & αφαίρεση διπλότυπων
duplicates = df.duplicated().sum()
print(f"Διπλότυπες εγγραφές: {duplicates:,} ({duplicates/len(df)*100:.2f}%)")

df.drop_duplicates(inplace=True)
print(f"Γραμμές μετά αφαίρεση διπλότυπων: {df.shape[0]:,}")

## 1.4 Ανάλυση Μεταβλητής Στόχου (Label)
Μελέτη κατανομής των κλάσεων και εντοπισμός ανισορροπίας. Ομαδοποίηση των 15 ετικετών σε 8 κατηγορίες επιθέσεων (DoS, DDoS, Brute Force, Web Attack, κ.λπ.).

In [ ]:
#Κατανομή Label
print("ΑΝΑΛΥΣΗ ΜΕΤΑΒΛΗΤΗΣ ΣΤΟΧΟΥ (Label)")


label_counts = df['Label'].value_counts()
label_pct = df['Label'].value_counts(normalize=True) * 100

label_summary = pd.DataFrame({'Πλήθος': label_counts, 'Ποσοστό (%)': label_pct.round(2)})
print(label_summary)
print(f"\nΣύνολο κλάσεων: {df['Label'].nunique()}")

In [ ]:
#Γραφήματα κατανομής Label
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Bar plot - όλες οι κλάσεις
label_counts.plot(kind='bar', ax=axes[0], color=sns.color_palette('viridis', len(label_counts)))
axes[0].set_title('Κατανομή Κλάσεων (Label)', fontsize=14)
axes[0].set_xlabel('Τύπος')
axes[0].set_ylabel('Πλήθος')
axes[0].tick_params(axis='x', rotation=45)

#chart μόνο επιθέσεις
attack_counts = label_counts[label_counts.index != 'BENIGN']
attack_counts.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', textprops={'fontsize': 9})
axes[1].set_title('Κατανομή Τύπων Επιθέσεων (χωρίς BENIGN)', fontsize=14)
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print("Παρατήρηση: Υπάρχει σημαντική ανισορροπία μεταξύ των κλάσεων.")
print(f"Η κλάση BENIGN αποτελεί το {label_pct['BENIGN']:.1f}% του dataset.")

In [ ]:
#Ομαδοποίηση επιθέσεων σε κατηγορίες
df['Attack Type'] = df['Label'].apply(lambda x:
    'BENIGN' if x == 'BENIGN' else
    'DDoS' if 'DDoS' in x else
    'DoS' if 'DoS' in x or 'Dos' in x else
    'Port Scan' if 'PortScan' in x else
    'Brute Force' if 'Patator' in x else
    'Web Attack' if 'Web Attack' in x else
    'Bot' if x == 'Bot' else
    'Infiltration' if x == 'Infiltration' else
    'Heartbleed' if x == 'Heartbleed' else x
)

print("Κατανομή ανά κατηγορία επίθεσης:")
print(df['Attack Type'].value_counts())

# Γράφημα
attack_only = df[df['Attack Type'] != 'BENIGN']
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

attack_only['Attack Type'].value_counts().plot(kind='bar', ax=axes[0], color=sns.color_palette('Set2'))
axes[0].set_title('Κατανομή Κατηγοριών Επιθέσεων (χωρίς BENIGN)', fontsize=13)
axes[0].set_ylabel('Πλήθος')
axes[0].tick_params(axis='x', rotation=45)

attack_only['Attack Type'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%')
axes[1].set_title('Ποσοστά Κατηγοριών Επιθέσεων', fontsize=13)
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

## 1.5 Στατιστική Περιγραφή & Οπτικοποιήσεις
Υπολογισμός βασικών στατιστικών μεγεθών (μέσος όρος, τυπική απόκλιση, skewness, kurtosis) και δημιουργία boxplots, histograms για τον εντοπισμό ακραίων τιμών και μοτίβων κατανομής.

In [ ]:
#Βασικά στατιστικά μεγέθη
numeric_df = df.select_dtypes(include=[np.number])

stats = numeric_df.describe().T
stats['median'] = numeric_df.median()
stats['skewness'] = numeric_df.skew()
stats['kurtosis'] = numeric_df.kurtosis()

print("=" * 60)
print("ΣΤΑΤΙΣΤΙΚΗ ΠΕΡΙΓΡΑΦΗ")
print("=" * 60)
display(stats[['count', 'mean', 'std', 'min', '25%', 'median', '75%', 'max', 'skewness', 'kurtosis']])

In [ ]:
#Boxplots βασικών χαρακτηριστικών
features_to_plot = ['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
                    'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean',
                    'Fwd Packet Length Mean', 'Bwd Packet Length Mean']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, feat in enumerate(features_to_plot):
    df[feat].plot(kind='box', ax=axes[i])
    axes[i].set_title(feat, fontsize=10)

plt.suptitle('Boxplots Βασικών Χαρακτηριστικών', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots ανά τύπο επίθεσης
selected_features = ['Flow Duration', 'Total Fwd Packets', 'Flow Bytes/s',
                     'Fwd Packet Length Mean', 'Bwd Packet Length Mean', 'Flow IAT Mean']

sample = df.sample(n=min(50000, len(df)), random_state=42)

for attack in sample['Attack Type'].unique():
    attack_data = sample[sample['Attack Type'] == attack][selected_features]
    plt.figure(figsize=(14, 5))
    sns.boxplot(data=attack_data, orient='h', palette='Set2')
    plt.title(f'Boxplot χαρακτηριστικών — {attack}', fontsize=14)
    plt.xlabel('Τιμή')
    plt.tight_layout()
    plt.show()

In [ ]:
#Histograms στηλών με λίγες unique τιμές (<50)
few_unique = [col for col in df.select_dtypes(include=[np.number]).columns
              if 1 < df[col].nunique() < 50]

print(f"Στήλες με <50 unique τιμές: {len(few_unique)}")

rows = (len(few_unique) + 3) // 4
fig, axes = plt.subplots(rows, 4, figsize=(20, 4 * rows))
axes = axes.flatten()

for i, col in enumerate(few_unique):
    df[col].value_counts().sort_index().plot(kind='bar', ax=axes[i], color='steelblue')
    axes[i].set_title(col, fontsize=9)
    axes[i].tick_params(axis='x', rotation=45, labelsize=7)

#Κρύψε κενά subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Κατανομές Στηλών με Λίγες Unique Τιμές', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Histograms κατανομών
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, feat in enumerate(features_to_plot):
    df[feat].hist(bins=50, ax=axes[i], color='steelblue', edgecolor='black')
    axes[i].set_title(feat, fontsize=10)

plt.suptitle('Κατανομές Βασικών Χαρακτηριστικών', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

### Ανάλυση Ακραίων Τιμών (Outliers)
Εντοπισμός outliers με τη μέθοδο IQR (Interquartile Range). Οι ακραίες τιμές δεν αφαιρούνται, καθώς στην ανίχνευση εισβολών οι επιθέσεις αποτελούν outlier συμπεριφορά.

In [ ]:
#Ανάλυση Outliers με IQR
numeric_df = df.select_dtypes(include=[np.number])

Q1 = numeric_df.quantile(0.25)
Q3 = numeric_df.quantile(0.75)
IQR = Q3 - Q1

outliers = (numeric_df < (Q1 - 1.5 * IQR)) | (numeric_df > (Q3 + 1.5 * IQR))
outlier_count = outliers.sum()
outlier_pct = (outliers.mean() * 100).round(2)

outlier_stats = pd.DataFrame({
    'Outlier Count': outlier_count,
    'Outlier Percentage (%)': outlier_pct
}).sort_values('Outlier Percentage (%)', ascending=False)

print("Ανάλυση Ακραίων Τιμών (IQR Method):")
display(outlier_stats)

# Γράφημα top 15 στηλών με τα περισσότερα outliers
top_outliers = outlier_stats.head(15)
plt.figure(figsize=(14, 6))
plt.bar(range(len(top_outliers)), top_outliers['Outlier Percentage (%)'], color='tomato')
plt.xticks(range(len(top_outliers)), top_outliers.index, rotation=45, ha='right')
plt.title('Top 15 Χαρακτηριστικά με τα Περισσότερα Outliers (%)', fontsize=14)
plt.ylabel('Ποσοστό Outliers (%)')
plt.tight_layout()
plt.show()

## 1.6 Ανάλυση Συσχετίσεων
Δημιουργία heatmap συσχετίσεων για όλα τα χαρακτηριστικά, εντοπισμός ζευγαριών με |r| > 0.9 και scatter plots. Υπολογισμός συσχέτισης κάθε χαρακτηριστικού με τη μεταβλητή στόχο (Label).

In [ ]:
#Heatmap συσχετίσεων
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

plt.figure(figsize=(24, 24))
sns.heatmap(corr_matrix, cmap='coolwarm', annot=False, linewidth=0.5)
plt.title('Heatmap Συσχετίσεων (όλα τα χαρακτηριστικά)', fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
#Συσχέτιση χαρακτηριστικών με Label
from sklearn.preprocessing import LabelEncoder

#Κωδικοποίηση Label σε αριθμό
le = LabelEncoder()
df['Label_encoded'] = le.fit_transform(df['Label'])

# Εμφάνιση αντιστοίχισης
print("Αντιστοίχιση κλάσεων:")
for val in sorted(df['Label_encoded'].unique()):
    print(f"  {val}: {le.inverse_transform([val])[0]}")

#Συσχέτιση κάθε χαρακτηριστικού με Label_encoded
numeric_df = df.select_dtypes(include=[np.number])
corr_with_label = numeric_df.corr()['Label_encoded'].drop('Label_encoded').sort_values(ascending=False)

print(f"\nΧαρακτηριστικά με θετική συσχέτιση με Label:")
print(corr_with_label[corr_with_label > 0].round(3))
print(f"\nΧαρακτηριστικά με αρνητική συσχέτιση με Label:")
print(corr_with_label[corr_with_label < 0].round(3))

#
plt.figure(figsize=(14, 8))
corr_with_label.sort_values().plot(kind='barh', color=['tomato' if x < 0 else 'steelblue' for x in corr_with_label.sort_values()])
plt.title('Συσχέτιση Χαρακτηριστικών με Label', fontsize=14)
plt.xlabel('Pearson Correlation')
plt.tight_layout()
plt.show()

# Καθαρισμός
df.drop('Label_encoded', axis=1, inplace=True)

In [ ]:
#Εντοπισμός υψηλά συσχετισμένων ζευγαριών
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

threshold = 0.9
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

high_corr = []
for col_idx, col in enumerate(upper.columns):
    for row in range(len(upper.index)):
        val = upper.iloc[row, col_idx]
        if abs(val) > threshold:
            high_corr.append((upper.index[row], col, round(val, 3)))

high_corr_df = pd.DataFrame(high_corr, columns=['Feature 1', 'Feature 2', 'Correlation'])
high_corr_df = high_corr_df.sort_values('Correlation', key=abs, ascending=False)

print(f"Ζεύγη με |συσχέτιση| > {threshold}: {len(high_corr_df)}")
display(high_corr_df)

In [ ]:
#Scatter plots υψηλά συσχετισμένων ζευγαριών (top 6)
top_pairs = high_corr_df.head(6)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

sample = df.sample(n=min(5000, len(df)), random_state=42)  # δείγμα για ταχύτητα

for i, (_, row) in enumerate(top_pairs.iterrows()):
    f1, f2, corr_val = row['Feature 1'], row['Feature 2'], row['Correlation']
    axes[i].scatter(sample[f1], sample[f2], alpha=0.3, s=5, color='steelblue')
    axes[i].set_xlabel(f1, fontsize=8)
    axes[i].set_ylabel(f2, fontsize=8)
    axes[i].set_title(f'r = {corr_val}', fontsize=10)

plt.suptitle('Scatter Plots Υψηλά Συσχετισμένων Ζευγαριών', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## 1.7 Επιλογή Χαρακτηριστικών (Feature Selection)
Αφαίρεση στηλών με: (α) σχεδόν μηδενική διακύμανση (std < 0.01), και (β) υψηλή συσχέτιση (|r| > 0.9) με άλλη στήλη. Επιλέχθηκε manual feature selection αντί PCA, ώστε να διατηρηθεί η ερμηνευσιμότητα των χαρακτηριστικών.

In [ ]:
#Feature Selection

print("FEATURE SELECTION")


columns_to_drop = []
reasons = {}

# Στήλες με μηδενική ή σχεδόν μηδενική διακύμανση
for col in numeric_df.columns:
    if numeric_df[col].std() < 0.01:
        columns_to_drop.append(col)
        reasons[col] = "Σχεδόν μηδενική διακύμανση"

# Αφαίρεση μίας στήλης από κάθε ζεύγος με |corr| > 0.9
corr_to_drop = set()
for _, row in high_corr_df.iterrows():
    if row['Feature 1'] not in corr_to_drop:
        corr_to_drop.add(row['Feature 2'])
        reasons[row['Feature 2']] = f"Υψηλή συσχέτιση ({row['Correlation']}) με {row['Feature 1']}"

columns_to_drop.extend(list(corr_to_drop))
columns_to_drop = list(set(columns_to_drop))

print(f"\nΣτήλες που αφαιρούνται ({len(columns_to_drop)}):\n")
for col in sorted(columns_to_drop):
    print(f"  ✗ {col}: {reasons.get(col, 'N/A')}")

# Αφαίρεση
df_clean = df.drop(columns=columns_to_drop, errors='ignore')
print(f"\n{'='*60}")
print(f"Διαστάσεις ΠΡΙΝ:  {df.shape[0]:,} γραμμές x {df.shape[1]} στήλες")
print(f"Διαστάσεις ΜΕΤΑ:  {df_clean.shape[0]:,} γραμμές x {df_clean.shape[1]} στήλες")
print(f"Αφαιρέθηκαν {len(columns_to_drop)} στήλες")
df = df_clean.copy()
print(f"\nΤελικό dataset: {df.shape[0]:,} γραμμές x {df.shape[1]} στήλες")

# Ερώτημα 2 — Κατηγοριοποίηση

## 2.1 Προετοιμασία Δεδομένων
Κωδικοποίηση της μεταβλητής στόχου (Label) σε αριθμητικές τιμές με LabelEncoder και κανονικοποίηση των χαρακτηριστικών με StandardScaler, ώστε να βρίσκονται στην ίδια κλίμακα.

In [ ]:
#Προετοιμασία δεδομένων
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Κωδικοποίηση Label
le = LabelEncoder()
df_clean['Label_encoded'] = le.fit_transform(df_clean['Label'])

print("Αντιστοίχιση κλάσεων:")
for val in sorted(df_clean['Label_encoded'].unique()):
    print(f"  {val}: {le.inverse_transform([val])[0]}")

#Χωρισμός σε X (features) και y (target)
X = df_clean.drop(columns=['Label', 'Attack Type', 'Label_encoded'])
y = df_clean['Label_encoded']

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nΚατανομή κλάσεων:")
print(y.value_counts())

# Κανονικοποίηση (StandardScaler)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)
print(f"\nΚανονικοποίηση ολοκληρώθηκε.")

## 2.2 Διαχείριση Ανισορροπίας Κλάσεων
Το dataset παρουσιάζει σοβαρή ανισορροπία (83% BENIGN). Χρησιμοποιούμε stratified sampling για διαχειρίσιμο μέγεθος και SMOTE (Synthetic Minority Over-sampling Technique) στο σύνολο εκπαίδευσης για εξισορρόπηση.Το SMOTE εφαρμόζεται μόνο στο train set, ώστε να μην υπάρχει data leakage.

In [ ]:
#Βήμα 2: Sampling & SMOTE
!pip install imbalanced-learn -q
from imblearn.over_sampling import SMOTE

#Χρήση Attack Type αντί Label (ομαδοποιημένες κατηγορίες)
y_grouped = df_clean['Attack Type']

#Αφαίρεση πολύ σπάνιων κλάσεων (<50 δείγματα)
class_counts = y_grouped.value_counts()
valid_classes = class_counts[class_counts >= 50].index
mask = y_grouped.isin(valid_classes)

X_filtered = X_scaled[mask]
y_filtered = y_grouped[mask]

print(f"Αφαιρέθηκαν κλάσεις με <50 δείγματα: {set(class_counts.index) - set(valid_classes)}")
print(f"Κλάσεις που κρατάμε: {sorted(valid_classes.tolist())}")

#δείγμα 100K
X_sample, _, y_sample, _ = train_test_split(
    X_filtered, y_filtered, train_size=100000, stratify=y_filtered, random_state=42
)

print(f"\nΔείγμα: {X_sample.shape[0]:,} γραμμές")
print(f"\nΚατανομή κλάσεων στο δείγμα:")
print(y_sample.value_counts())

## 2.3 Σενάριο Α: Train/Validation/Test Split
Διαχωρισμός δεδομένων σε 60% εκπαίδευση, 20% επικύρωση, 20% τελικός έλεγχος. Εφαρμογή SMOTE μόνο στο σύνολο εκπαίδευσης. Βελτιστοποίηση υπερπαραμέτρων μέσω Grid Search στο validation set. Αξιολόγηση στο test set.

In [ ]:
#Σενάριο Α: Train/Validation/Test Split (60/20/20)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

#Split: 60% train, 20% validation, 20% test
X_train_full, X_test_A, y_train_full, y_test_A = train_test_split(
    X_sample, y_sample, test_size=0.2, stratify=y_sample, random_state=42
)
X_train_A, X_val_A, y_train_A, y_val_A = train_test_split(
    X_train_full, y_train_full, test_size=0.25, stratify=y_train_full, random_state=42
)

print(f"Train:      {X_train_A.shape[0]:,} ({X_train_A.shape[0]/len(X_sample)*100:.0f}%)")
print(f"Validation: {X_val_A.shape[0]:,} ({X_val_A.shape[0]/len(X_sample)*100:.0f}%)")
print(f"Test:       {X_test_A.shape[0]:,} ({X_test_A.shape[0]/len(X_sample)*100:.0f}%)")

#SMOTE μόνο στο train set
smote = SMOTE(random_state=42)
X_train_A_sm, y_train_A_sm = smote.fit_resample(X_train_A, y_train_A)

print(f"\nTrain μετά SMOTE: {X_train_A_sm.shape[0]:,}")
print(f"Κατανομή μετά SMOTE:")
print(pd.Series(y_train_A_sm).value_counts())

### Grid Search — Σενάριο Α
Βελτιστοποίηση υπερπαραμέτρων για κάθε αλγόριθμο, αξιολογώντας στο validation set. Εκπαίδευση στο train set (μετά SMOTE) και επιλογή του καλύτερου συνδυασμού παραμέτρων.

In [ ]:
# Σενάριο Α: Grid Search
from itertools import product
import time

def grid_search_val(model_class, param_grid, X_tr, y_tr, X_v, y_v):
    best_score = 0
    best_params = None
    best_model = None

    keys = list(param_grid.keys())
    values = list(param_grid.values())
    combos = list(product(*values))
    total = len(combos)

    for i, combo in enumerate(combos, 1):
        params = dict(zip(keys, combo))
        print(f"  [{i}/{total}] {params}", end=" ... ", flush=True)
        start = time.time()
        model = model_class(**params, random_state=42)
        model.fit(X_tr, y_tr)
        score = f1_score(y_v, model.predict(X_v), average='weighted')
        elapsed = time.time() - start
        print(f"F1={score:.4f} ({elapsed:.1f}s)")

        if score > best_score:
            best_score = score
            best_params = params
            best_model = model

    return best_model, best_params, best_score

#Logistic Regression
print("1. LOGISTIC REGRESSION")

lr_params = {
    'C': [0.1, 1, 10],
    'max_iter': [1000],
    'solver': ['lbfgs']
}
lr_best_A, lr_params_A, lr_score_A = grid_search_val(
    LogisticRegression, lr_params, X_train_A_sm, y_train_A_sm, X_val_A, y_val_A
)
print(f">>> Καλύτερο: {lr_params_A} | F1={lr_score_A:.4f}")

#Decision Tree

print("2. DECISION TREE")

dt_params = {
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 10],
    'criterion': ['gini']
}
dt_best_A, dt_params_A, dt_score_A = grid_search_val(
    DecisionTreeClassifier, dt_params, X_train_A_sm, y_train_A_sm, X_val_A, y_val_A
)
print(f"Καλύτερο: {dt_params_A} | F1={dt_score_A:.4f}")

#Random Forest

print("3. RANDOM FOREST")

rf_params = {
    'n_estimators': [50, 100],
    'max_depth': [10, 15],
    'min_samples_split': [2, 5]
}
rf_best_A, rf_params_A, rf_score_A = grid_search_val(
    RandomForestClassifier, rf_params, X_train_A_sm, y_train_A_sm, X_val_A, y_val_A
)
print(f"Καλύτερο: {rf_params_A} | F1={rf_score_A:.4f}")

### Αξιολόγηση Σενάριο Α — Test Set
Αξιολόγηση των βέλτιστων μοντέλων στο test set με accuracy, precision, recall, F1-score και confusion matrix.

In [ ]:
#Αξιολόγηση Σενάριο Α στο Test Set
models_A = {
    'Logistic Regression': lr_best_A,
    'Decision Tree': dt_best_A,
    'Random Forest': rf_best_A
}

results_A = {}

for name, model in models_A.items():
    y_pred = model.predict(X_test_A)

    acc = accuracy_score(y_test_A, y_pred)
    prec = precision_score(y_test_A, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test_A, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test_A, y_pred, average='weighted', zero_division=0)

    results_A[name] = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1-Score': f1}

    print(f"\n{'=' * 50}")
    print(f"{name}")
    print(f"{'=' * 50}")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"\nClassification Report:")
    print(classification_report(y_test_A, y_pred, zero_division=0))

    # Confusion Matrix
    fig, ax = plt.subplots(figsize=(8, 6))
    cm = confusion_matrix(y_test_A, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=model.classes_).plot(ax=ax, cmap='Blues', values_format='d')
    ax.set_title(f'Confusion Matrix — {name} (Σενάριο Α)', fontsize=13)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 2.4 Σενάριο Β: Grid Search με Cross-Validation
Διαχωρισμός 80% εκπαίδευση / 20% τελικός έλεγχος. Εφαρμογή SMOTE στο train set και βελτιστοποίηση υπερπαραμέτρων μέσω GridSearchCV με 5-fold cross-validation. Αξιολόγηση στο test set.

In [ ]:
#Σενάριο Β: Train/Test Split (80/20) + GridSearchCV
from sklearn.model_selection import GridSearchCV

#Split 80/20
X_train_B, X_test_B, y_train_B, y_test_B = train_test_split(
    X_sample, y_sample, test_size=0.2, stratify=y_sample, random_state=42
)

#SMOTE στο train set
smote_B = SMOTE(random_state=42)
X_train_B_sm, y_train_B_sm = smote_B.fit_resample(X_train_B, y_train_B)

print(f"Train: {X_train_B.shape[0]:,} → μετά SMOTE: {X_train_B_sm.shape[0]:,}")
print(f"Test:  {X_test_B.shape[0]:,}")

# 1.Logistic Regression
print(f"\n{'=' * 50}")
print("1. LOGISTIC REGRESSION (GridSearchCV)")
print("=" * 50)
lr_grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    {'C': [0.1, 1, 10], 'solver': ['lbfgs']},
    cv=5, scoring='f1_weighted', n_jobs=-1, verbose=1
)
lr_grid.fit(X_train_B_sm, y_train_B_sm)
print(f"Καλύτερες παράμετροι: {lr_grid.best_params_}")
print(f"Καλύτερο CV F1: {lr_grid.best_score_:.4f}")

# 2.Decision Tree
print(f"\n{'=' * 50}")
print("2. DECISION TREE (GridSearchCV)")
print("=" * 50)
dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    {'max_depth': [5, 10, 15], 'min_samples_split': [2, 10], 'criterion': ['gini']},
    cv=5, scoring='f1_weighted', n_jobs=-1, verbose=1
)
dt_grid.fit(X_train_B_sm, y_train_B_sm)
print(f"Καλύτερες παράμετροι: {dt_grid.best_params_}")
print(f"Καλύτερο CV F1: {dt_grid.best_score_:.4f}")

# 3.Random Forest
print(f"\n{'=' * 50}")
print("3. RANDOM FOREST (GridSearchCV)")
print("=" * 50)
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {'n_estimators': [50, 100], 'max_depth': [10, 15], 'min_samples_split': [2, 5]},
    cv=5, scoring='f1_weighted', n_jobs=-1, verbose=1
)
rf_grid.fit(X_train_B_sm, y_train_B_sm)
print(f"Καλύτερες παράμετροι: {rf_grid.best_params_}")
print(f"Καλύτερο CV F1: {rf_grid.best_score_:.4f}")

### Αξιολόγηση Σενάριο Β — Test Set
Αξιολόγηση των βέλτιστων μοντέλων (από GridSearchCV) στο test set.

In [ ]:
#Αξιολόγηση Σενάριο Β στο Test Set
models_B = {
    'Logistic Regression': lr_grid.best_estimator_,
    'Decision Tree': dt_grid.best_estimator_,
    'Random Forest': rf_grid.best_estimator_
}

results_B = {}

for name, model in models_B.items():
    y_pred = model.predict(X_test_B)

    acc = accuracy_score(y_test_B, y_pred)
    prec = precision_score(y_test_B, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test_B, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test_B, y_pred, average='weighted', zero_division=0)

    results_B[name] = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1-Score': f1}

    print(f"\n{'=' * 50}")
    print(f"{name}")
    print(f"{'=' * 50}")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"\nClassification Report:")
    print(classification_report(y_test_B, y_pred, zero_division=0))

    #Confusion Matrix
    fig, ax = plt.subplots(figsize=(8, 6))
    cm = confusion_matrix(y_test_B, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=model.classes_).plot(ax=ax, cmap='Blues', values_format='d')
    ax.set_title(f'Confusion Matrix — {name} (Σενάριο Β)', fontsize=13)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 2.5 Συγκριτική Ανάλυση Σεναρίων & Αλγορίθμων
Σύγκριση των δύο σεναρίων (Train/Val/Test vs Cross-Validation) και των τριών αλγορίθμων ως προς accuracy, precision, recall και F1-score.

In [ ]:

comparison = pd.DataFrame({
    ('Σενάριο Α', 'LR'): results_A['Logistic Regression'],
    ('Σενάριο Α', 'DT'): results_A['Decision Tree'],
    ('Σενάριο Α', 'RF'): results_A['Random Forest'],
    ('Σενάριο Β', 'LR'): results_B['Logistic Regression'],
    ('Σενάριο Β', 'DT'): results_B['Decision Tree'],
    ('Σενάριο Β', 'RF'): results_B['Random Forest'],
}).T

comparison.columns = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
display(comparison.round(4))

#σύγκριση
fig, axes = plt.subplots(1, 2, figsize=(18, 6))


metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics))
width = 0.12

for i, (idx, row) in enumerate(comparison.iterrows()):
    label = f"{idx[0]} - {idx[1]}"
    axes[0].bar(x + i * width, row.values, width, label=label)

axes[0].set_xticks(x + width * 2.5)
axes[0].set_xticklabels(metrics)
axes[0].set_ylim(0.85, 1.01)
axes[0].set_title('Σύγκριση Μετρικών ανά Μοντέλο & Σενάριο', fontsize=13)
axes[0].legend(fontsize=8)
axes[0].grid(axis='y', alpha=0.3)

#F1-Score σύγκριση
f1_data = pd.DataFrame({
    'Σενάριο Α': [results_A[m]['F1-Score'] for m in ['Logistic Regression', 'Decision Tree', 'Random Forest']],
    'Σενάριο Β': [results_B[m]['F1-Score'] for m in ['Logistic Regression', 'Decision Tree', 'Random Forest']]
}, index=['LR', 'DT', 'RF'])

f1_data.plot(kind='bar', ax=axes[1], color=['steelblue', 'tomato'])
axes[1].set_title('F1-Score: Σενάριο Α vs Σενάριο Β', fontsize=13)
axes[1].set_ylim(0.85, 1.01)
axes[1].set_ylabel('F1-Score')
axes[1].tick_params(axis='x', rotation=0)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


best_model_A = max(results_A, key=lambda x: results_A[x]['F1-Score'])
best_model_B = max(results_B, key=lambda x: results_B[x]['F1-Score'])
print(f"Καλύτερο μοντέλο Σενάριο Α: {best_model_A} (F1={results_A[best_model_A]['F1-Score']:.4f})")
print(f"Καλύτερο μοντέλο Σενάριο Β: {best_model_B} (F1={results_B[best_model_B]['F1-Score']:.4f})")
print(f"\nΤο Σενάριο Β (Cross-Validation) παρέχει πιο αξιόπιστη εκτίμηση")
print(f"καθώς αξιολογεί το μοντέλο σε 5 διαφορετικά folds.")

# Ερώτημα 3 — Clustering

## 3.1 Προετοιμασία Δεδομένων
Χρήση του κανονικοποιημένου dataset από το Ερώτημα 2 (χωρίς Label)

In [ ]:
#Καθαρισμός μνήμης
import gc

#Διαγραφή μεταβλητών Ερωτήματος 2 που δεν χρειάζονται πια
for var in ['X_train_A', 'X_val_A', 'X_test_A', 'y_train_A', 'y_val_A', 'y_test_A',
            'X_train_A_sm', 'y_train_A_sm', 'X_train_B', 'X_test_B', 'y_train_B', 'y_test_B',
            'X_train_B_sm', 'y_train_B_sm', 'X_train_full', 'y_train_full',
            'lr_best_A', 'dt_best_A', 'rf_best_A', 'lr_grid', 'dt_grid', 'rf_grid',
            'X_sample', 'y_sample']:
    if var in globals():
        del globals()[var]

gc.collect()
print("Μνήμη καθαρίστηκε! Κρατήθηκαν: X_filtered, y_filtered, X_scaled")

In [ ]:
#Clustering
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, normalized_mutual_info_score
from sklearn.neighbors import NearestNeighbors
import time

#Δείγμα 10K
X_clust, _, y_clust_labels, _ = train_test_split(
    X_filtered, y_filtered, train_size=10000, stratify=y_filtered, random_state=42
)

#PCA 10 components για clustering
pca_10 = PCA(n_components=10, random_state=42)
X_clust_pca = pca_10.fit_transform(X_clust)
print(f"PCA 10 components: {pca_10.explained_variance_ratio_.sum()*100:.1f}% διακύμανσης")

#PCA 2D μόνο για οπτικοποίηση
pca_2d = PCA(n_components=2, random_state=42)
X_pca_2d = pca_2d.fit_transform(X_clust)
print(f"PCA 2 components: {pca_2d.explained_variance_ratio_.sum()*100:.1f}% διακύμανσης (μόνο για visualization)")

print(f"\nΔείγμα: {X_clust_pca.shape[0]:,} γραμμές")
print(f"Κλάσεις:\n{y_clust_labels.value_counts()}")

#Οπτικοποίηση πραγματικών κλάσεων
plt.figure(figsize=(10, 7))
for label in sorted(y_clust_labels.unique()):
    mask = y_clust_labels == label
    plt.scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1], label=label, s=5, alpha=0.5)
plt.title('PCA — Πραγματικές Κλάσεις (Attack Type)', fontsize=14)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(markerscale=3, fontsize=9)
plt.tight_layout()
plt.show()

## 3.2 K-Means Clustering
Δοκιμή διαφορετικών τιμών k (2-10). Χρήση Elbow Method και Silhouette Score για επιλογή βέλτιστου k.

In [ ]:
#K-Means: Elbow Method & Silhouette Analysis (k=2-20)
inertias = []
sil_scores = []
db_scores = []
K_range = range(2, 21)

for k in K_range:
    print(f"  K={k}", end=" ... ", flush=True)
    start = time.time()
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_clust_pca)
    sil = silhouette_score(X_clust_pca, labels)
    db = davies_bouldin_score(X_clust_pca, labels)
    inertias.append(km.inertia_)
    sil_scores.append(sil)
    db_scores.append(db)
    print(f"Silhouette={sil:.4f}, Davies-Bouldin={db:.4f} ({time.time()-start:.1f}s)")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_title('Elbow Method', fontsize=13)
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inertia')
axes[1].plot(K_range, sil_scores, 'go-')
axes[1].set_title('Silhouette Score', fontsize=13)
axes[1].set_xlabel('k')
axes[1].set_ylabel('Score')
axes[2].plot(K_range, db_scores, 'ro-')
axes[2].set_title('Davies-Bouldin Index', fontsize=13)
axes[2].set_xlabel('k')
axes[2].set_ylabel('Score (χαμηλότερο = καλύτερο)')
plt.tight_layout()
plt.show()

best_k = list(K_range)[np.argmax(sil_scores)]
print(f"\nΒέλτιστο k (Silhouette): {best_k} (score={max(sil_scores):.4f})")

km_best = KMeans(n_clusters=best_k, random_state=42, n_init=10)
km_labels = km_best.fit_predict(X_clust_pca)

plt.figure(figsize=(10, 7))
for c in range(best_k):
    mask = km_labels == c
    plt.scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1], label=f'Cluster {c}', s=5, alpha=0.5)
plt.title(f'K-Means (k={best_k}) — PCA', fontsize=14)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(markerscale=3)
plt.tight_layout()
plt.show()

## 3.3 Hierarchical Clustering
Δοκιμή τριών μεθόδων σύνδεσης (ward, complete, average) με αριθμό clusters ίσο με τις πραγματικές κλάσεις (7) και με τον βέλτιστο k=2 από K-Means.

In [ ]:
# Hierarchical Clustering
linkage_methods = ['ward', 'complete', 'average']
n_clusters_list = [2, 7]

hc_results = {}

for n_cl in n_clusters_list:
    for link in linkage_methods:
        print(f"  n={n_cl}, linkage={link}", end=" ... ", flush=True)
        start = time.time()
        hc = AgglomerativeClustering(n_clusters=n_cl, linkage=link)
        labels = hc.fit_predict(X_clust_pca)
        sil = silhouette_score(X_clust_pca, labels)
        db = davies_bouldin_score(X_clust_pca, labels)
        hc_results[(n_cl, link)] = {'labels': labels, 'sil': sil, 'db': db}
        print(f"Silhouette={sil:.4f}, DB={db:.4f} ({time.time()-start:.1f}s)")

best_hc_key = max(hc_results, key=lambda x: hc_results[x]['sil'])
hc_labels = hc_results[best_hc_key]['labels']
print(f"\nΚαλύτερο: n={best_hc_key[0]}, linkage={best_hc_key[1]} (Sil={hc_results[best_hc_key]['sil']:.4f})")

plt.figure(figsize=(10, 7))
for c in range(best_hc_key[0]):
    mask = hc_labels == c
    plt.scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1], label=f'Cluster {c}', s=5, alpha=0.5)
plt.title(f'Hierarchical ({best_hc_key[1]}, n={best_hc_key[0]}) — PCA', fontsize=14)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(markerscale=3)
plt.tight_layout()
plt.show()

## 3.4 DBSCAN
Δοκιμή διαφορετικών τιμών epsilon και min_samples. Ο DBSCAN εντοπίζει αυτόματα τον αριθμό clusters και αναγνωρίζει θόρυβο (noise points).

In [ ]:
# DBSCAN: Εύρεση βέλτιστου epsilon με KNN Distance Plot
nn = NearestNeighbors(n_neighbors=5)
nn.fit(X_clust_pca)
distances, _ = nn.kneighbors(X_clust_pca)
distances = np.sort(distances[:, 4])

plt.figure(figsize=(12, 5))
plt.plot(distances)
plt.title('KNN Distance Plot (k=5) — Εύρεση epsilon', fontsize=14)
plt.xlabel('Σημεία (ταξινομημένα)')
plt.ylabel('Απόσταση 5ου γείτονα')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Δοκιμή τιμών epsilon βάσει του γραφήματος
eps_values = [1, 2, 3, 4, 5, 7]
min_samples_values = [5, 10, 15]

dbscan_results = {}

for eps in eps_values:
    for ms in min_samples_values:
        print(f"  eps={eps}, min_samples={ms}", end=" ... ", flush=True)
        start = time.time()
        db_model = DBSCAN(eps=eps, min_samples=ms)
        labels = db_model.fit_predict(X_clust_pca)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = (labels == -1).sum()

        if n_clusters >= 2:
            sil = silhouette_score(X_clust_pca, labels)
            dbi = davies_bouldin_score(X_clust_pca[labels != -1], labels[labels != -1])
        else:
            sil = -1
            dbi = -1

        dbscan_results[(eps, ms)] = {'labels': labels, 'n_clusters': n_clusters,
                                      'noise': n_noise, 'sil': sil, 'db': dbi}
        print(f"clusters={n_clusters}, noise={n_noise}, Sil={sil:.4f} ({time.time()-start:.1f}s)")

# Καλύτερο DBSCAN
valid = {k: v for k, v in dbscan_results.items() if v['sil'] > 0 and v['n_clusters'] >= 2}
if valid:
    best_db_key = max(valid, key=lambda x: valid[x]['sil'])
    db_labels = dbscan_results[best_db_key]['labels']
    print(f"\nΚαλύτερο: eps={best_db_key[0]}, min_samples={best_db_key[1]}")
    print(f"  Clusters={dbscan_results[best_db_key]['n_clusters']}, Noise={dbscan_results[best_db_key]['noise']}, Sil={dbscan_results[best_db_key]['sil']:.4f}")

    plt.figure(figsize=(10, 7))
    for c in sorted(set(db_labels)):
        mask = db_labels == c
        name = f'Cluster {c}' if c != -1 else 'Noise'
        plt.scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1], label=name, s=5, alpha=0.5)
    plt.title(f'DBSCAN (eps={best_db_key[0]}, min_s={best_db_key[1]}) — PCA', fontsize=14)
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.legend(markerscale=3)
    plt.tight_layout()
    plt.show()

## 3.5 Clustering μόνο στις Επιθέσεις
Αφαιρώντας το BENIGN traffic, εξετάζουμε αν οι αλγόριθμοι μπορούν να διαχωρίσουν τους διαφορετικούς τύπους επιθέσεων μεταξύ τους.

In [ ]:
# Clustering μόνο στις επιθέσεις
mask_attacks = y_clust_labels != 'BENIGN'
X_attacks = X_clust_pca[mask_attacks]
y_attacks = y_clust_labels[mask_attacks]
X_pca_attacks = X_pca_2d[mask_attacks]

n_attack_types = y_attacks.nunique()
print(f"Δείγμα επιθέσεων: {X_attacks.shape[0]:,}")
print(f"Τύποι επιθέσεων: {n_attack_types}")
print(f"Κλάσεις:\n{y_attacks.value_counts()}")

# K-Means στις επιθέσεις
km_attack = KMeans(n_clusters=n_attack_types, random_state=42, n_init=10)
km_attack_labels = km_attack.fit_predict(X_attacks)
sil_att = silhouette_score(X_attacks, km_attack_labels)
db_att = davies_bouldin_score(X_attacks, km_attack_labels)
nmi_att = normalized_mutual_info_score(y_attacks, km_attack_labels)
print(f"\nK-Means (k={n_attack_types}) στις επιθέσεις:")
print(f"  Silhouette: {sil_att:.4f}")
print(f"  Davies-Bouldin: {db_att:.4f}")
print(f"  NMI: {nmi_att:.4f}")

# Οπτικοποίηση: πραγματικές vs clusters
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for label in sorted(y_attacks.unique()):
    mask = y_attacks == label
    axes[0].scatter(X_pca_attacks[mask, 0], X_pca_attacks[mask, 1], label=label, s=10, alpha=0.6)
axes[0].set_title('Πραγματικοί Τύποι Επιθέσεων', fontsize=13)
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].legend(markerscale=2)

for c in range(n_attack_types):
    mask = km_attack_labels == c
    axes[1].scatter(X_pca_attacks[mask, 0], X_pca_attacks[mask, 1], label=f'Cluster {c}', s=10, alpha=0.6)
axes[1].set_title(f'K-Means (k={n_attack_types}) — Μόνο Επιθέσεις', fontsize=13)
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].legend(markerscale=2)

plt.tight_layout()
plt.show()

# Crosstab
print("\nΑντιστοίχιση Clusters με Τύπους Επιθέσεων:")
ct_attacks = pd.crosstab(y_attacks, km_attack_labels, margins=True)
ct_attacks.columns = [f'Cluster {c}' if c != 'All' else 'All' for c in ct_attacks.columns]
display(ct_attacks)

## 3.6 Συγκριτική Αξιολόγηση & Σύγκριση με Label
Σύγκριση των τριών αλγορίθμων clustering ως προς Silhouette Score, Davies-Bouldin Index, NMI και Purity. Αντιστοίχιση clusters με πραγματικές κλάσεις (Label) μέσω crosstab.

In [ ]:
# Συνάρτηση Purity
def purity_score(y_true, y_pred):
    from collections import Counter
    clusters = set(y_pred)
    total = 0
    for c in clusters:
        if c == -1:
            continue
        mask = y_pred == c
        counter = Counter(y_true[mask])
        total += counter.most_common(1)[0][1]
    return total / sum(1 for x in y_pred if x != -1)

# Μετρικές για κάθε αλγόριθμο
print("=" * 70)
print("ΣΥΓΚΡΙΤΙΚΟΣ ΠΙΝΑΚΑΣ ΑΛΓΟΡΙΘΜΩΝ CLUSTERING")
print("=" * 70)

km_sil = silhouette_score(X_clust_pca, km_labels)
km_db = davies_bouldin_score(X_clust_pca, km_labels)
km_nmi = normalized_mutual_info_score(y_clust_labels, km_labels)
km_pur = purity_score(y_clust_labels.values, km_labels)

hc_sil = hc_results[best_hc_key]['sil']
hc_dbi = hc_results[best_hc_key]['db']
hc_nmi = normalized_mutual_info_score(y_clust_labels, hc_labels)
hc_pur = purity_score(y_clust_labels.values, hc_labels)

db_sil = dbscan_results[best_db_key]['sil']
db_dbi = dbscan_results[best_db_key]['db']
db_nmi = normalized_mutual_info_score(y_clust_labels, db_labels)
db_pur = purity_score(y_clust_labels.values, db_labels)

comp_df = pd.DataFrame({
    'Αλγόριθμος': ['K-Means', 'Hierarchical', 'DBSCAN'],
    'Παράμετροι': [f'k={best_k}', f'n={best_hc_key[0]}, {best_hc_key[1]}', f'eps={best_db_key[0]}, min_s={best_db_key[1]}'],
    'Clusters': [best_k, best_hc_key[0], dbscan_results[best_db_key]['n_clusters']],
    'Silhouette': [round(km_sil, 4), round(hc_sil, 4), round(db_sil, 4)],
    'Davies-Bouldin': [round(km_db, 4), round(hc_dbi, 4), round(db_dbi, 4)],
    'NMI': [round(km_nmi, 4), round(hc_nmi, 4), round(db_nmi, 4)],
    'Purity': [round(km_pur, 4), round(hc_pur, 4), round(db_pur, 4)]
})
display(comp_df)

# Γραφήματα σύγκρισης
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].bar(comp_df['Αλγόριθμος'], comp_df['Silhouette'], color=['steelblue', 'tomato', 'green'])
axes[0,0].set_title('Silhouette (↑ καλύτερο)', fontsize=12)
axes[0,0].set_ylim(0, 1)

axes[0,1].bar(comp_df['Αλγόριθμος'], comp_df['Davies-Bouldin'], color=['steelblue', 'tomato', 'green'])
axes[0,1].set_title('Davies-Bouldin (↓ καλύτερο)', fontsize=12)

axes[1,0].bar(comp_df['Αλγόριθμος'], comp_df['NMI'], color=['steelblue', 'tomato', 'green'])
axes[1,0].set_title('NMI (↑ καλύτερο)', fontsize=12)
axes[1,0].set_ylim(0, 1)

axes[1,1].bar(comp_df['Αλγόριθμος'], comp_df['Purity'], color=['steelblue', 'tomato', 'green'])
axes[1,1].set_title('Purity (↑ καλύτερο)', fontsize=12)
axes[1,1].set_ylim(0, 1)

plt.suptitle('Σύγκριση Αλγορίθμων Clustering', fontsize=15)
plt.tight_layout()
plt.show()

# Σύγκριση με πραγματικές κλάσεις
print("\n" + "=" * 70)
print("ΣΥΓΚΡΙΣΗ CLUSTERS ΜΕ ΠΡΑΓΜΑΤΙΚΕΣ ΚΛΑΣΕΙΣ")
print("=" * 70)

print("\n--- K-Means ---")
ct_km = pd.crosstab(y_clust_labels, km_labels, margins=True)
ct_km.columns = [f'Cluster {c}' if c != 'All' else 'All' for c in ct_km.columns]
display(ct_km)

print("\n--- Hierarchical ---")
ct_hc = pd.crosstab(y_clust_labels, hc_labels, margins=True)
ct_hc.columns = [f'Cluster {c}' if c != 'All' else 'All' for c in ct_hc.columns]
display(ct_hc)

print("\n--- DBSCAN ---")
ct_db = pd.crosstab(y_clust_labels, db_labels, margins=True)
ct_db.columns = [f'Cluster {c}' if c != 'All' else 'All' for c in ct_db.columns]
display(ct_db)